In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
METADATA_PATH = PROJECT_ROOT / "data" / "fma_metadata" / "tracks.csv"

df = pd.read_csv(METADATA_PATH, header=[0, 1], low_memory=False)

print("Shape:", df.shape)

Shape: (106575, 53)


In [2]:
columns = pd.DataFrame(df.columns.tolist(), columns=["level_1", "level_2"])

columns

,level_1,level_2
0,Unnamed: 0_level_0,Unnamed: 0_level_1
1,album,comments
2,album,date_created
3,album,date_released
4,album,engineer
5,album,favorites
6,album,id
7,album,information
8,album,listens
9,album,producer


In [3]:
Subset_distribution = (df.set.subset.value_counts())

Subset_distribution

subset
large     81574
medium    17000
small      8000
Name: count, dtype: int64

In [4]:
genre_distribution = df.track.genre_top.value_counts(dropna=False)

genre_distribution

genre_top
NaN                    56977
Rock                   14182
Experimental           10608
Electronic              9372
Hip-Hop                 3552
Folk                    2803
Pop                     2332
Instrumental            2079
International           1389
Classical               1230
Jazz                     571
Old-Time / Historic      554
Spoken                   423
Country                  194
Soul-RnB                 175
Blues                    110
Easy Listening            24
Name: count, dtype: int64

In [5]:
small = df[df.set.subset == "small"]

print("Number of tracks:", len(small))

split_distribution = small.set.split.value_counts()

split_distribution

Number of tracks: 8000


split
training      6400
validation     800
test           800
Name: count, dtype: int64

In [6]:
missing_values = small.isnull().sum()

missing_values

Unnamed: 0_level_0  Unnamed: 0_level_1       0
album               comments                 0
                    date_created             0
                    date_released         2663
                    engineer              6858
                    favorites                0
                    id                       0
                    information           1498
                    listens                  0
                    producer              6596
                    tags                     0
                    title                    0
                    tracks                   0
                    type                   231
artist              active_year_begin     6458
                    active_year_end       7642
                    associated_labels     7151
                    bio                   2086
                    comments                 0
                    date_created             0
                    favorites                0
             

In [7]:
tag_count = (small.track.tags != "[]").sum()
print("tracks with tags:", tag_count)

tagged_tracks = small[small.track.tags != "[]"]
tagged_tracks.track.tags.head(20)

tracks with tags: 1361


743                                         ['baltimore']
975                                         ['baltimore']
1548    ['avant garde', 'experimental', 'noise', 'san ...
3308                                     ['in her dream']
3364                                               ['il']
3373                                               ['il']
3381                                               ['il']
3421                                               ['il']
3977    ['new york city', 'punk', 'blues rock', 'garag...
5500                                             ['folk']
5501                                             ['folk']
5502                                             ['folk']
5503                                             ['folk']
5504                                             ['folk']
5505                                             ['folk']
5506                                             ['folk']
5507                                             ['folk']
5508          

In [8]:
from ast import literal_eval

all_tags = []

for tags in tagged_tracks.track.tags:
    all_tags.extend(literal_eval(tags))

tag_counts = pd.Series(all_tags).value_counts()

print("unique tags:", len(tag_counts))

tag_counts.head(30)

unique tags: 1191


electronic                    145
instrumental                  140
experimental                   98
acoustic                       84
electronica                    75
soundtrack                     71
ambient                        70
folk                           66
tracks to sync                 63
pop                            52
dub                            48
dance                          44
rock                           44
children                       42
kid-friendly                   42
horror                         41
piano                          41
childrens music                40
guitar                         40
for kids                       38
hip hop                        38
electropop                     37
shoegaze                       35
irish                          35
psychedelic                    35
techno                         33
ireland                        33
holiday music                  32
background                     30
various artist

In [9]:
top_tags = tag_counts.head(50)

top_tags

electronic                    145
instrumental                  140
experimental                   98
acoustic                       84
electronica                    75
soundtrack                     71
ambient                        70
folk                           66
tracks to sync                 63
pop                            52
dub                            48
dance                          44
rock                           44
children                       42
kid-friendly                   42
horror                         41
piano                          41
childrens music                40
guitar                         40
for kids                       38
hip hop                        38
electropop                     37
shoegaze                       35
irish                          35
psychedelic                    35
techno                         33
ireland                        33
holiday music                  32
background                     30
various artist

In [10]:
tracks_with_top_tags = 0

for i in tagged_tracks.track.tags:
    i = literal_eval(i)

    for j in i:
        if j in top_tags.index:
            tracks_with_top_tags += 1
            break

print("tracks with at least one top-50 tag:", tracks_with_top_tags)

tracks with at least one top-50 tag: 895


In [11]:
tag_labels = []

for i in tagged_tracks.track.tags:
    i = literal_eval(i)
    j = [k for k in i if k in top_tags.index]

    if len(j) > 0:
        tag_labels.append(j)

print("number of usable tracks:", len(tag_labels))
print("example labels:", tag_labels[:10])

number of usable tracks: 895
example labels: [['experimental', 'noise'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk']]


In [12]:
tag_labels = []

for i in tagged_tracks.track.tags:
    i = literal_eval(i)
    
    j = []
    for k in i:
        if k in top_tags.index:
            j.append(k)
    
    if len(j) > 0:
        tag_labels.append(j)

print("number of usable tracks:", len(tag_labels))
print("example labels:", tag_labels[:10])

number of usable tracks: 895
example labels: [['experimental', 'noise'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk'], ['folk']]


In [13]:
task1_data = []

for i in tagged_tracks.track.tags:
    i = literal_eval(i)

    if len([j for j in i if j in top_tags.index]) > 0:
        task1_data.append(i)
    else:
        pass

print("number of tracks:", len(task1_data))

number of tracks: 895


In [14]:
text_columns = small[['artist', 'album', 'track']]

text_columns.head()

artist                  \
      active_year_begin active_year_end   
1   2006-01-01 00:00:00             NaN   
3   2006-01-01 00:00:00             NaN   
4                   NaN             NaN   
16  1999-01-01 00:00:00             NaN   
17  1999-01-01 00:00:00             NaN   

                                                       \
                                    associated_labels   
1                                                 NaN   
3                                                 NaN   
4   Mexican Summer, Richie Records, Woodsist, Skul...   
16                                                NaN   
17                                                NaN   

                                                                \
                                                  bio comments   
1   <p>A Way Of Life, A Collective of Hip-Hop from...      0.0   
3   <p>A Way Of Life, A Collective of Hip-Hop from...      0.0   
4   <p><span style="font-family:Verdana, Geneva, A...      3.0   
16  <p>The Eyesores originally formed in 1997 orig...      0.0   
17  <p>The Eyesores originally formed in 1997 orig...      0.0   

                                                                    ...  \
           date_created favorites    id   latitude        location  ...   
1   2008-11-26 01:42:32       9.0   1.0  40.058324      New Jersey  ...   
3   2008-11-26 01:42:32       9.0   1.0  40.058324      New Jersey  ...   
4   2008-11-26 01:42:55      74.0   6.0        NaN             NaN  ...   
16  2008-11-26 01:47:44      11.0  54.0  41.823989  Providence, RI  ...   
17  2008-11-26 01:47:44      11.0  54.0  41.823989  Providence, RI  ...   

         track                         \
   information interest language_code   
1          NaN   4656.0            en   
3          NaN   1933.0            en   
4          NaN  54881.0            en   
16         NaN   1593.0            en   
17         NaN    839.0            en   

                                                                         \
                                              license  listens lyricist   
1   Attribution-NonCommercial-ShareAlike 3.0 Inter...   1293.0      NaN   
3   Attribution-NonCommercial-ShareAlike 3.0 Inter...   1151.0      NaN   
4   Attribution-NonCommercial-NoDerivatives (aka M...  50135.0      NaN   
16  Attribution-Noncommercial-No Derivative Works ...   1299.0      NaN   
17  Attribution-Noncommercial-No Derivative Works ...    725.0      NaN   

                                              
   number publisher tags               title  
1     3.0       NaN   []                Food  
3     6.0       NaN   []          This World  
4     1.0       NaN   []             Freeway  
16    2.0       NaN   []  Queen Of The Wires  
17    4.0       NaN   []                Ohio  

[5 rows x 50 columns]

In [15]:
artist_bio = small.artist.bio

artist_bio.head(10)

1     <p>A Way Of Life, A Collective of Hip-Hop from...
3     <p>A Way Of Life, A Collective of Hip-Hop from...
4     <p><span style="font-family:Verdana, Geneva, A...
16    <p>The Eyesores originally formed in 1997 orig...
17    <p>The Eyesores originally formed in 1997 orig...
23    <p>Power Electronics redefined by Andy Ortmann...
56    <p>Ariel "Pink" Rosenberg has been churning ou...
62    <p><i><span></span></i>Ed Askew cut one of the...
65    <p><i><span></span></i>Ed Askew cut one of the...
66    <p><i><span></span></i>Ed Askew cut one of the...
Name: bio, dtype: object

In [16]:
bio_count = artist_bio.notna().sum()

print("tracks with artist bio:", bio_count)

tracks with artist bio: 5914


In [17]:
track_information = small.track.information.head(10)
track_title = small.track.title.head(10) 
track_tags = small.track.tags.head(10)

print("track information:\n", track_information)
print()
print("track title:\n", track_title)
print()
print("track tags:\n", track_tags)

track information:
 1     NaN
3     NaN
4     NaN
16    NaN
17    NaN
23    NaN
56    NaN
62    NaN
65    NaN
66    NaN
Name: information, dtype: object

track title:
 1                      Food
3                This World
4                   Freeway
16       Queen Of The Wires
17                     Ohio
23               Blackout 2
56    Jules Lost His Jewels
62          Castle Of Stars
65            Here With You
66         All I want to Do
Name: title, dtype: object

track tags:
 1     []
3     []
4     []
16    []
17    []
23    []
56    []
62    []
65    []
66    []
Name: tags, dtype: object
